## Regressão Logística

A [Regressão Logística](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) é um modelo de classificação supervisionada amplamente utilizado em problemas binários.

O modelo funciona a partir de uma combinação linear das variáveis de entrada, produzindo um valor que é transformado em probabilidade por meio da função logística (sigmoid), resultando em valores entre 0 e 1.

### Hiperparâmetros utilizados

- **C**: controla a regularização do modelo.
  - Valores menores (ex: 0.01) → modelo mais simples (mais regularização)
  - Valores maiores (ex: 10) → modelo mais complexo (menos regularização)
  - Os valores foram escolhidos em escala logarítmica para explorar diferentes ordens de magnitude.

- **solver**: algoritmo de otimização utilizado para encontrar os coeficientes do modelo.
  - `lbfgs`: bom desempenho geral, especialmente para datasets maiores (default)
  - `liblinear`: eficiente para problemas binários e datasets menores

### Estratégia

Foi utilizado o GridSearchCV com validação cruzada (5 folds) para identificar a melhor combinação de hiperparâmetros, utilizando a métrica ROC AUC, adequada para problemas com classes desbalanceadas.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from preprocessing import load_and_preprocess

X_train, X_test, y_train, y_test = load_and_preprocess(
    "../predictive_models/scrdata_202505.csv"
)

param_grid_lr = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["lbfgs", "liblinear"]
}

grid_lr = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, class_weight="balanced"),
    param_grid=param_grid_lr,
    cv=5,
    scoring="roc_auc",
    n_jobs=2
)

grid_lr.fit(X_train, y_train)

print("Melhores parâmetros:", grid_lr.best_params_)
print("Melhor score (CV):", grid_lr.best_score_)

best_lr = grid_lr.best_estimator_

y_pred = best_lr.predict(X_test)
y_proba = best_lr.predict_proba(X_test)[:, 1]

roc = roc_auc_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)

resultados = {
    "modelo": "LogisticRegression",
    "roc_auc": roc,
    "f1": f1,
    "accuracy": acc,
    "roc_auc_cv": grid_lr.best_score_,
    "best_params": str(grid_lr.best_params_)
}

df_result = pd.DataFrame([resultados])
df_result.to_csv("resultados_lr.csv", index=False, sep=";")

   uf_AL  uf_AM  uf_AP  uf_BA  uf_CE  uf_DF  uf_ES  uf_GO  uf_MA  uf_MG  ...  \
2      0      0      0      0      0      0      0      0      0      0  ...   
3      0      0      0      0      0      0      0      0      0      0  ...   
4      0      0      0      0      0      0      0      0      0      0  ...   
5      0      0      0      0      0      0      0      0      0      0  ...   
6      0      0      0      0      0      0      0      0      0      0  ...   

   submodalidade_Microcrédito  \
2                           0   
3                           0   
4                           0   
5                           0   
6                           0   

   submodalidade_Outros com característica de crédito  \
2                                                  0    
3                                                  0    
4                                                  0    
5                                                  0    
6                                  